# pipeline

> Predifined sklearn pipeline to process and create features with financial data

In [ ]:
#| default_exp pipeline

In [ ]:
#| hide
from eccore.ipython import nb_setup, pandas_nrows_ncols
from myquantlab.core import load_test_df
from nbdev import nbdev_export

In [ ]:
#| hide
nb_setup()

Set autoreload mode


In [ ]:
#| export
import re

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin, BaseEstimator
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler

## Custom transforms

Transforms applicable to preprocess and feature engineer financial data.

In [ ]:
#| export
class MyBaseTransformer(BaseEstimator, TransformerMixin):
    """Base class for my custom transformers"""
    def __init__(self):
        self.input_features_ = None
        self.output_features_ = None
        self.postfix = "transformed"

    def fit(self, X, y=None):
        if hasattr(X, 'columns'):
            self.input_features_ = X.columns.tolist()
        else:
            self.input_features_ = [f'f_{i}' for i in range(X.shape[1])]
        self.output_features_ = [f"{feat}_{self.postfix}" for feat in self.input_features_]
        return self

    def transform(self, X) -> np.ndarray:
        X = X.copy()
        return X.values

    def get_feature_names_out(self, input_features=None) -> list[str]|None:
        if input_features is not None:
            return [f"{feat}_{self.postfix}" for feat in input_features]
        else:
            return self.output_features_

In [ ]:
pipe = Pipeline([
    ('base-transformer', MyBaseTransformer())
])

df = load_test_df()
pipe.fit_transform(df)[:5,:]

array([[ 2759.02,  2779.27,  2747.27,  2754.48, 26562.  ],
       [ 2753.11,  2755.36,  2690.69,  2743.45, 38777.  ],
       [ 2744.83,  2748.58,  2651.23,  2672.8 , 41777.  ],
       [ 2670.8 ,  2722.9 ,  2657.93,  2680.71, 39034.  ],
       [ 2675.59,  2692.34,  2627.59,  2663.57, 61436.  ]])

In [ ]:
pd.DataFrame(data=pipe.fit_transform(df), columns=pipe.get_feature_names_out(), index=df.index).head(5)

,Open_transformed,High_transformed,Low_transformed,Close_transformed,Volume_transformed
dt,,,,,
2018-10-22,2759.02,2779.27,2747.27,2754.48,26562.0
2018-10-23,2753.11,2755.36,2690.69,2743.45,38777.0
2018-10-24,2744.83,2748.58,2651.23,2672.80,41777.0
2018-10-25,2670.80,2722.90,2657.93,2680.71,39034.0
2018-10-26,2675.59,2692.34,2627.59,2663.57,61436.0


In [ ]:
#| export
class SimpleReturnTransformer(MyBaseTransformer):
    """Evaluate the percentage return over 1 or more periods    """
    def __init__(self, periods:int=1) -> None:
        super().__init__()
        self.periods = periods
        self.postfix = f"ret{self.periods}"
        self.nickname = f"R{self.periods}"

    def transform(self, X) -> np.ndarray:
        """percentage change with previous bar, fist bar is 0"""
        X = X.copy()
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)
        return X.pct_change(periods=self.periods).fillna(0.0).values

This pipeline computes simple return: $R_t = \Large \frac{P_{t-1}-P_t}{P_t} \normalsize = \Large \frac{P_{t-1}}{P_t}- \normalsize 1$

In [ ]:
pipe = ColumnTransformer([
    ('R', SimpleReturnTransformer(), ['Open', 'Close', 'Volume'])
])
pd.DataFrame(pipe.fit_transform(df), columns=pipe.get_feature_names_out(), index=df.index).head(5)

,R__Open_ret1,R__Close_ret1,R__Volume_ret1
dt,,,
2018-10-22,0.000000,0.000000,0.000000
2018-10-23,-0.002142,-0.004004,0.459867
2018-10-24,-0.003008,-0.025752,0.077365
2018-10-25,-0.026971,0.002959,-0.065658
2018-10-26,0.001793,-0.006394,0.573910


In [ ]:
#| export
class LogReturnTransformer(MyBaseTransformer):
    """Evaluate the logarithmic return over 1 or more periods
    """
    def __init__(self, periods:int=1) -> None:
        super().__init__()
        self.periods = periods
        self.postfix = f"logr{self.periods}"
        self.nickname = f"r{self.periods}"

    def transform(self, X) -> np.ndarray:
        """log-return with previous bar, first bar is 0"""
        X = X.copy()
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)
        return np.log1p(X.pct_change(periods=self.periods).fillna(0.0)).values


This pipeline computes Log Return: $r_t = \ln(\Large\frac{P_{t}}{P_{t-1}}\normalsize)$


In [ ]:
pipe = ColumnTransformer([
    ('r', LogReturnTransformer(), ['Open', 'Close', 'Volume'])
])
pd.DataFrame(pipe.fit_transform(df), columns=pipe.get_feature_names_out(), index=df.index).head(5)

,r__Open_logr1,r__Close_logr1,r__Volume_logr1
dt,,,
2018-10-22,0.000000,0.000000,0.000000
2018-10-23,-0.002144,-0.004012,0.378346
2018-10-24,-0.003012,-0.026090,0.074519
2018-10-25,-0.027341,0.002955,-0.067913
2018-10-26,0.001792,-0.006414,0.453563


In [ ]:
pipe = ColumnTransformer([
    ('R', SimpleReturnTransformer(), ['Open', 'Close', 'Volume']),
    ('r', LogReturnTransformer(), ['Open', 'Close', 'Volume'])
])
pd.DataFrame(pipe.fit_transform(df), columns=pipe.get_feature_names_out(), index=df.index).head(5)

,R__Open_ret1,R__Close_ret1,R__Volume_ret1,r__Open_logr1,r__Close_logr1,r__Volume_logr1
dt,,,,,,
2018-10-22,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2018-10-23,-0.002142,-0.004004,0.459867,-0.002144,-0.004012,0.378346
2018-10-24,-0.003008,-0.025752,0.077365,-0.003012,-0.026090,0.074519
2018-10-25,-0.026971,0.002959,-0.065658,-0.027341,0.002955,-0.067913
2018-10-26,0.001793,-0.006394,0.573910,0.001792,-0.006414,0.453563


In [ ]:
#| export
class StdTransformer(MyBaseTransformer):
    """Evaluate the standard deviation over a window"""
    def __init__(self, window:int=5) -> None:
        super().__init__()
        self.window = window
        self.postfix = f"std{self.window}"
        self.nickname = f"std{self.window}"

    def transform(self, X) -> np.ndarray:
        X = X.copy()
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)
        return X.rolling(window=self.window, min_periods=1).std().values

In [ ]:
pipe = ColumnTransformer([
    ('std', StdTransformer(3), ['Open', 'Close'])
])
pd.DataFrame(data=pipe.fit_transform(df), columns=pipe.get_feature_names_out(), index=df.index).head(5)


,std__Open_std3,std__Close_std3
dt,,
2018-10-22,NaN,NaN
2018-10-23,4.179001,7.799388
2018-10-24,7.127910,44.318367
2018-10-25,45.320958,38.708953
2018-10-26,41.427774,8.578467


In [ ]:
#| export
class MATransformer(MyBaseTransformer):
    """Evaluate the moving average over a window"""
    def __init__(self, window:int=5) -> None:
        super().__init__()
        self.window = window
        self.postfix = f"MA{self.window}"
        self.nickname = f"ma{self.window}"

    def transform(self, X) -> np.ndarray:
        X = X.copy()
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)
        return X.rolling(window=self.window, min_periods=1).mean().values

In [ ]:
#| export
class EMATransformer(MyBaseTransformer):
    """Evaluate the exponential moving average over a window"""
    def __init__(self, window:int=5) -> None:
        super().__init__()
        self.window = window
        self.postfix = f"EMA{self.window}"
        self.nickname = f"ema{self.window}"

    def transform(self, X) -> np.ndarray:
        X = X.copy()
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)
        return X.ewm(span=self.window).mean().values

Build a pipeline applying these transforms to specific columns.

In [ ]:
pipe = ColumnTransformer([
    ('thru', 'passthrough', ['Open', 'High', 'Low', 'Close', 'Volume']),
    ('ret', SimpleReturnTransformer(3), ['Close']),
    ('ma', MATransformer(3), ['Close']),
    ('ema', EMATransformer(3), ['Open', 'Close'])
])
pd.DataFrame(data=pipe.fit_transform(df), columns=pipe.get_feature_names_out(), index=df.index).head(5)


,thru__Open,thru__High,thru__Low,thru__Close,thru__Volume,ret__Close_ret3,ma__Close_MA3,ema__Open_EMA3,ema__Close_EMA3
dt,,,,,,,,,
2018-10-22,2759.02,2779.27,2747.27,2754.48,26562.0,0.000000,2754.480000,2759.020000,2754.480000
2018-10-23,2753.11,2755.36,2690.69,2743.45,38777.0,0.000000,2748.965000,2755.080000,2747.126667
2018-10-24,2744.83,2748.58,2651.23,2672.80,41777.0,0.000000,2723.576667,2749.222857,2704.654286
2018-10-25,2670.80,2722.90,2657.93,2680.71,39034.0,-0.026782,2698.986667,2707.397333,2691.884000
2018-10-26,2675.59,2692.34,2627.59,2663.57,61436.0,-0.029117,2672.360000,2690.980645,2677.270323


In [ ]:
#| export
def simplify_colnames(cols)->list[str]:
    """Simplify the columns names by removing the prefix"""
    pat = re.compile(r"[\w\d\-]*_{2}(?P<end>\w*)")
    cols = [pat.match(c).group('end') for c in cols]
    return cols

In [ ]:
print(pipe.get_feature_names_out())

['thru__Open' 'thru__High' 'thru__Low' 'thru__Close' 'thru__Volume'
 'ret__Close_ret3' 'ma__Close_MA3' 'ema__Open_EMA3' 'ema__Close_EMA3']


In [ ]:
print(simplify_colnames(pipe.get_feature_names_out()))

['Open', 'High', 'Low', 'Close', 'Volume', 'Close_ret3', 'Close_MA3', 'Open_EMA3', 'Close_EMA3']


In [ ]:
df_proc = pd.DataFrame(
    data=pipe.fit_transform(df), 
    columns=simplify_colnames(pipe.get_feature_names_out()), 
    index=df.index)
df_proc.head(5)

,Open,High,Low,Close,Volume,Close_ret3,Close_MA3,Open_EMA3,Close_EMA3
dt,,,,,,,,,
2018-10-22,2759.02,2779.27,2747.27,2754.48,26562.0,0.000000,2754.480000,2759.020000,2754.480000
2018-10-23,2753.11,2755.36,2690.69,2743.45,38777.0,0.000000,2748.965000,2755.080000,2747.126667
2018-10-24,2744.83,2748.58,2651.23,2672.80,41777.0,0.000000,2723.576667,2749.222857,2704.654286
2018-10-25,2670.80,2722.90,2657.93,2680.71,39034.0,-0.026782,2698.986667,2707.397333,2691.884000
2018-10-26,2675.59,2692.34,2627.59,2663.57,61436.0,-0.029117,2672.360000,2690.980645,2677.270323


In [ ]:
#| export
def dframe(
    df:pd.DataFrame,    # dataframe to transform 
    pipe:BaseEstimator,      # pipeline to apply on the dataframe
    fit:bool=True       # fit_transform if True, transform if False
    ) -> pd.DataFrame:  # formated DataFrame
    """Takes a pipeline, (optionaly) fit it on df and transform it, then return a well formated DataFrame"""
    if fit:
        data = pipe.fit_transform(df)
    elif not fit:
        data = pipe.transform(df)
    else:
        raise ValueError("fit must be a boolean")

    df_proc = pd.DataFrame(
        data=data, 
        columns=simplify_colnames(pipe.get_feature_names_out()),
        index=df.index)
    return df_proc

In [ ]:
dframe(df, pipe).head(5)

,Open,High,Low,Close,Volume,Close_ret3,Close_MA3,Open_EMA3,Close_EMA3
dt,,,,,,,,,
2018-10-22,2759.02,2779.27,2747.27,2754.48,26562.0,0.000000,2754.480000,2759.020000,2754.480000
2018-10-23,2753.11,2755.36,2690.69,2743.45,38777.0,0.000000,2748.965000,2755.080000,2747.126667
2018-10-24,2744.83,2748.58,2651.23,2672.80,41777.0,0.000000,2723.576667,2749.222857,2704.654286
2018-10-25,2670.80,2722.90,2657.93,2680.71,39034.0,-0.026782,2698.986667,2707.397333,2691.884000
2018-10-26,2675.59,2692.34,2627.59,2663.57,61436.0,-0.029117,2672.360000,2690.980645,2677.270323


## Complex Pipeline Factory

input: OHLCV 

parameters: 

- feature creation:
    - `coi`: list of columns to apply the feature engineering on
    - `transforms`: list of transforms to apply on each selected column

- stats applied to new features
    - `stats`: list of stats to apply on new feature


In [ ]:
coi = ['Open', 'Close']
transforms = [SimpleReturnTransformer(1), SimpleReturnTransformer(10)]
stats = [EMATransformer(5), EMATransformer(20), EMATransformer(60), StdTransformer(5)]

In [ ]:
#| export
def build_pipeline(
    coi:list[str],                  # list of columns to use for feature engineering 
    transforms:list[BaseEstimator], # list of feature engineering transformers to apply to each column in coi
    stats: list[BaseEstimator]      # list of stats transformers to apply to engineered feature
    ) -> BaseEstimator :
    """Build a pipeline using cols in `coi`, transformers in `transforms` and stats in `stats`

    The output will be a pipeline that takes a OHLCV DataFrame and returns:

     - the original OHLCV columns
     - for the columns in `coi`, engineers features by applying each of the pipelines in `transforms`
     - then adds the statistics in `stats` to each of the engineered features
    """
    feature_stats_steps = [('thru', 'passthrough')]
    for stat in stats:
        feature_stats_steps.append((stat.nickname, stat))   #type:ignore
    feature_stats = FeatureUnion(feature_stats_steps)       #type:ignore
    feature_stats.nickname = 'stats'                        #type:ignore
    
    feature_engineering = [Pipeline(
        [(t.nickname, t),(t.nickname+feature_stats.nickname, feature_stats)] #type:ignore
        ) for t in transforms]
    pipe_steps = [('thru', 'passthrough', ['Open', 'High', 'Low', 'Close', 'Volume'])]

    for feat in feature_engineering:
        pipe_steps.append((feat.steps[0][0], feat, coi))

    pipe = ColumnTransformer(pipe_steps)

    return pipe

In [ ]:
pipe = build_pipeline(coi, transforms, stats)

pipe

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('thru', ...), ('R1', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e

In [ ]:
with pandas_nrows_ncols():
    display(dframe(df, pipe).head(5))

,Open,High,Low,Close,Volume,Open_ret1,Close_ret1,Open_ret1_EMA5,Close_ret1_EMA5,Open_ret1_EMA20,Close_ret1_EMA20,Open_ret1_EMA60,Close_ret1_EMA60,Open_ret1_std5,Close_ret1_std5,Open_ret10,Close_ret10,Open_ret10_EMA5,Close_ret10_EMA5,Open_ret10_EMA20,Close_ret10_EMA20,Open_ret10_EMA60,Close_ret10_EMA60,Open_ret10_std5,Close_ret10_std5
dt,,,,,,,,,,,,,,,,,,,,,,,,,
2018-10-22,2759.02,2779.27,2747.27,2754.48,26562.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
2018-10-23,2753.11,2755.36,2690.69,2743.45,38777.0,-0.002142,-0.004004,-0.001285,-0.002403,-0.001125,-0.002102,-0.001089,-0.002036,0.001515,0.002832,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2018-10-24,2744.83,2748.58,2651.23,2672.80,41777.0,-0.003008,-0.025752,-0.002101,-0.013463,-0.001816,-0.010786,-0.001750,-0.010206,0.001548,0.013858,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2018-10-25,2670.80,2722.90,2657.93,2680.71,39034.0,-0.026971,0.002959,-0.012432,-0.006641,-0.009078,-0.006818,-0.008374,-0.006748,0.012690,0.013019,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2018-10-26,2675.59,2692.34,2627.59,2663.57,61436.0,0.001793,-0.006394,-0.006971,-0.006546,-0.006448,-0.006716,-0.006203,-0.006673,0.011836,0.011275,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
#| hide
import nbdev
nbdev.nbdev_export()